### Here we'll prove that our batch ingestion is idempotent.

```text
First run:
CSV files → COPY INTO → Bronze
                         ↓
                    13717 records

Second run:
Same CSV files → COPY INTO → Bronze
                         ↓
                    Still 13717 records
```
It should not become 27434

In [0]:
-- 1. Check current record count
SELECT
    COUNT(*) AS before_count
FROM retail_lakehouse.bronze.orders;

before_count
13717


In [0]:
-- 2. Run COPY INTO again
COPY INTO retail_lakehouse.bronze.orders
FROM (
  SELECT
    CAST(order_id AS BIGINT) AS order_id,
    CAST(customer_id AS BIGINT) AS customer_id,
    CAST(order_date AS DATE) AS order_date,
    product_id,
    CAST(quantity AS INT) AS quantity,
    CAST(unit_price AS DECIMAL(10,2)) AS unit_price,
    status
  FROM '/Volumes/retail_lakehouse/raw/retail_files/orders/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true'
);

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


In [0]:
-- 3. Check record count after second execution
SELECT
    COUNT(*) AS after_count
FROM retail_lakehouse.bronze.orders;

after_count
13717


**And That was our idempotency proof.**

Now the below code will show you the duplicate records in the dataset not idempotency failuer

In [0]:
-- 4. Check duplicate order IDs
SELECT
    order_id,
    COUNT(*) AS record_count
FROM retail_lakehouse.bronze.orders
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY record_count DESC;

order_id,record_count
499,2


In [0]:
-- Idempotency validation
SELECT
    CASE
        WHEN COUNT(*) = 13717
        THEN 'PASS - Idempotent ingestion verified'
        ELSE 'FAIL - Unexpected record count'
    END AS validation_result
FROM retail_lakehouse.bronze.orders;

validation_result
PASS - Idempotent ingestion verified


In [0]:
DESCRIBE HISTORY retail_lakehouse.bronze.orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-08-29T07:05:27.000Z,72492180296752,datatoinfo03@gmail.com,COPY INTO,Map(statsOnLoad -> true),null,List(3404234639386653),0a82bb70-429c-4e81-869a-82910544721a,0829-063324-e56qsdze-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 13717, numOutputBytes -> 72947, numSkippedCorruptFiles -> 0)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
0,2026-08-29T06:45:59.000Z,72492180296752,datatoinfo03@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-f7d8b691-4587-4987-b409-f48569d587b6"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-cb3d7ad8-9065-46fd-975d-d1beebbb5de2""}, statsOnLoad -> false)",null,List(3404234639386653),92e3f90d-21d9-438a-ba37-ecaeb13547a7,0829-063324-e56qsdze-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13


_**COPY INTO tracks the source files that have already been processed. If the same job is rerun with the same files, those files are skipped rather than inserted again. In our project we loaded 13,717 orders, reran the same COPY INTO command, and verified that the target still contained 13,717 records.**_